# LangChain RAG 逐步复现
## 使用方式

1. 建议先在当前目录打开这个 Notebook。
2. 优先运行前面的“环境检查”单元。
3. 如果没有配置 `DASHSCOPE_API_KEY` 或没有启动 `Ollama`，相关章节会自动跳过并提示原因。
4. 可以按顺序逐段学习，也可以直接点击“全部运行”。

## 推荐依赖安装命令

```bash
python -m pip install -U langchain langchain-core langchain-community langchain-text-splitters langchain-ollama langchain-chroma chromadb dashscope jq pypdf
```


In [ ]:
#直接在notebook中安装依赖环境
%pip install -U langchain langchain-core langchain-community langchain-text-splitters langchain-ollama langchain-chroma chromadb dashscope jq pypdf

## 00. 环境检查与公共配置

- 这一节不对应单独的 `.py` 文件，专门用来让后面的案例更稳地执行。
- 会检查当前目录、数据目录、`DASHSCOPE_API_KEY`、`Ollama` 服务端口和若干可选依赖。
- 后续章节如果前置条件不足，会显示“跳过原因”而不是直接报错。


In [ ]:
from __future__ import annotations

import importlib.util
import os
import socket
from pathlib import Path


def locate_project_root() -> Path:
    signatures = [("data/stu.csv", "data/info.csv", "data/Python基础语法.txt")]
    seen: set[Path] = set()
    candidates: list[Path] = []

    def add_candidate(path: Path) -> None:
        resolved = path.resolve()
        if resolved not in seen and resolved.exists() and resolved.is_dir():
            seen.add(resolved)
            candidates.append(resolved)

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        add_candidate(base)
        try:
            child_dirs = [child for child in base.iterdir() if child.is_dir()]
        except PermissionError:
            child_dirs = []
        for child in child_dirs:
            add_candidate(child)
            try:
                grandchild_dirs = [grandchild for grandchild in child.iterdir() if grandchild.is_dir()]
            except PermissionError:
                grandchild_dirs = []
            for grandchild in grandchild_dirs:
                add_candidate(grandchild)

    for candidate in candidates:
        if all((candidate / rel).exists() for rel in signatures[0]):
            return candidate

    raise FileNotFoundError("没有找到 02_LangChainRAG开发 目录，请确认从仓库目录中打开 Notebook。")


ROOT_DIR = locate_project_root()
os.chdir(ROOT_DIR)
DATA_DIR = ROOT_DIR / "data"
CHAT_HISTORY_DIR = ROOT_DIR / "chat_history"
CHROMA_DIR = ROOT_DIR / "chroma_db"
CHAT_HISTORY_DIR.mkdir(exist_ok=True)
CHROMA_DIR.mkdir(exist_ok=True)


def is_port_open(host: str = "127.0.0.1", port: int = 11434, timeout: float = 1.0) -> bool:
    sock = socket.socket()
    sock.settimeout(timeout)
    try:
        sock.connect((host, port))
        return True
    except OSError:
        return False
    finally:
        sock.close()


HAS_DASHSCOPE = bool(os.getenv("DASHSCOPE_API_KEY"))
HAS_OLLAMA = is_port_open()
HAS_JQ = importlib.util.find_spec("jq") is not None
HAS_PYPDF = importlib.util.find_spec("pypdf") is not None


def section_ready(section_id: str, condition: bool, reason: str) -> bool:
    if condition:
        print(f"[运行] {section_id}")
        return True
    print(f"[跳过] {section_id}: {reason}")
    return False


print(f"ROOT_DIR = {ROOT_DIR}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"DASHSCOPE_API_KEY = {'已配置' if HAS_DASHSCOPE else '未配置'}")
print(f"Ollama(11434) = {'可连接' if HAS_OLLAMA else '不可连接'}")
print(f"jq = {'已安装' if HAS_JQ else '未安装'}")
print(f"pypdf = {'已安装' if HAS_PYPDF else '未安装'}")


## 01. 01[扩展]余弦相似度

- 对应源码：`01[扩展]余弦相似度.py`
- 本节说明：用最基础的方式理解向量点积、模长与余弦相似度，为后面的嵌入与检索打底。
- 运行提示：直接可运行，无需额外前置条件。


In [ ]:
import numpy as np

"""
计算两个向量的余弦相似度（衡量方向相似性，剔除长度影响）
"""


def get_dot(vec_a, vec_b):
    """计算 2 个向量的点积。"""
    if len(vec_a) != len(vec_b):
        raise ValueError("2个向量必须维度数量相同")

    dot_sum = 0
    for a, b in zip(vec_a, vec_b):
        dot_sum += a * b

    return dot_sum


def get_norm(vec):
    """计算单个向量的模长。"""
    sum_square = 0
    for v in vec:
        sum_square += v * v

    return np.sqrt(sum_square)


def cosine_similarity(vec_a, vec_b):
    """余弦相似度：点积 / 模长乘积。"""
    return get_dot(vec_a, vec_b) / (get_norm(vec_a) * get_norm(vec_b))


if __name__ == "__main__":
    vec_a = [0.5, 0.5]
    vec_b = [0.7, 0.7]
    vec_c = [0.7, 0.5]
    vec_d = [-0.6, -0.5]

    print("ab:", cosine_similarity(vec_a, vec_b))
    print("ac:", cosine_similarity(vec_a, vec_c))
    print("ad:", cosine_similarity(vec_a, vec_d))


## 02. 02LangChain访问阿里云通义千问大模型

- 对应源码：`02LangChain访问阿里云通义千问大模型.py`
- 本节说明：演示如何通过 LangChain 调用阿里云通义千问文本模型，这一节需要 `DASHSCOPE_API_KEY`。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("02", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.llms import Tongyi

    # qwen-max 适合作为文本补全模型使用
    model = Tongyi(model="qwen-max")

    res = model.invoke("你是谁呀，能做什么？")
    print(res)


## 03. 03LangChain访问Ollama本地模型

- 对应源码：`03LangChain访问Ollama本地模型.py`
- 本节说明：演示如何连接本地 Ollama 大模型，这一节要求本机 `Ollama` 已启动并已拉取对应模型。
- 运行提示：运行前置：未检测到本地 Ollama 服务（默认端口 11434）。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("03", HAS_OLLAMA, "未检测到本地 Ollama 服务（默认端口 11434）"):
    from langchain_ollama import OllamaLLM

    model = OllamaLLM(model="qwen3:4b")

    res = model.invoke("你是谁呀，能做什么？")
    print(res)


## 04. 04LangChain的流式输出

- 对应源码：`04LangChain的流式输出.py`
- 本节说明：对比云端模型和本地模型的流式输出效果，便于理解 `stream()` 的用法。
- 运行提示：直接可运行，无需额外前置条件。


In [ ]:
print("## 通义千问流式输出")
if section_ready("04-A", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.llms import Tongyi

    model = Tongyi(model="qwen-max")
    for chunk in model.stream("你是谁呀，能做什么？"):
        print(chunk, end="", flush=True)
    print("\n")

print("## Ollama 流式输出")
if section_ready("04-B", HAS_OLLAMA, "未检测到本地 Ollama 服务（默认端口 11434）"):
    from langchain_ollama import OllamaLLM

    model = OllamaLLM(model="qwen3:4b")
    for chunk in model.stream("你是谁呀，能做什么？"):
        print(chunk, end="", flush=True)
    print()


## 05. 05LangChain调用聊天模型

- 对应源码：`05LangChain调用聊天模型.py`
- 本节说明：演示消息列表形式的聊天模型调用，重点观察 `System / Human / AI` 三类消息如何协作。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("05", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

    model = ChatTongyi(model="qwen3-max")

    messages = [
        SystemMessage(content="你是一个边塞诗人。"),
        HumanMessage(content="写一首唐诗。"),
        AIMessage(content="锄禾日当午，汗滴禾下土，谁知盘中餐，粒粒皆辛苦。"),
        HumanMessage(content="按照你上一个回复的格式，再写一首唐诗。"),
    ]

    for chunk in model.stream(messages):
        print(chunk.content, end="", flush=True)


## 06. 06LangChain调用Ollama的聊天模型

- 对应源码：`06LangChain调用Ollama的聊天模型.py`
- 本节说明：把上一节的聊天模型调用方式切换到本地 Ollama 版本。
- 运行提示：运行前置：未检测到本地 Ollama 服务（默认端口 11434）。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("06", HAS_OLLAMA, "未检测到本地 Ollama 服务（默认端口 11434）"):
    from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
    from langchain_ollama import ChatOllama

    model = ChatOllama(model="qwen3:4b")

    messages = [
        SystemMessage(content="你是一个边塞诗人。"),
        HumanMessage(content="写一首唐诗。"),
        AIMessage(content="锄禾日当午，汗滴禾下土，谁知盘中餐，粒粒皆辛苦。"),
        HumanMessage(content="按照你上一个回复的格式，再写一首唐诗。"),
    ]

    for chunk in model.stream(messages):
        print(chunk.content, end="", flush=True)


## 07. 07LangChain消息的简写形式

- 对应源码：`07LangChain消息的简写形式.py`
- 本节说明：说明 LangChain 中消息也可以用元组简写，写法更紧凑。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("07", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi

    model = ChatTongyi(model="qwen3-max")

    messages = [
        ("system", "你是一个边塞诗人。"),
        ("human", "写一首唐诗。"),
        ("ai", "锄禾日当午，汗滴禾下土，谁知盘中餐，粒粒皆辛苦。"),
        ("human", "按照你上一个回复的格式，再写一首唐诗。"),
    ]

    for chunk in model.stream(messages):
        print(chunk.content, end="", flush=True)


## 08. 08LangChain访问阿里云嵌入模型

- 对应源码：`08LangChain访问阿里云嵌入模型.py`
- 本节说明：演示查询向量与批量文档向量的生成，后面做向量检索都会依赖这个能力。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("08", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.embeddings import DashScopeEmbeddings

    model = DashScopeEmbeddings(model="text-embedding-v4")

    print(model.embed_query("我喜欢你"))
    print(model.embed_documents(["我喜欢你", "我稀饭你", "晚上吃啥"]))


## 09. 09LangChain访问Ollama的本地嵌入模型

- 对应源码：`09LangChain访问Ollama的本地嵌入模型.py`
- 本节说明：将嵌入模型切换为本地 Ollama 版本，便于离线场景实验。
- 运行提示：运行前置：未检测到本地 Ollama 服务（默认端口 11434）。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("09", HAS_OLLAMA, "未检测到本地 Ollama 服务（默认端口 11434）"):
    from langchain_ollama import OllamaEmbeddings

    model = OllamaEmbeddings(model="qwen3-embedding:4b")

    print(model.embed_query("我喜欢你"))
    print(model.embed_documents(["我喜欢你", "我稀饭你", "晚上吃啥"]))


## 10. 10通用提示词模板

- 对应源码：`10通用提示词模板.py`
- 本节说明：先从最常见的 `PromptTemplate` 入手，体会变量填充和链式调用。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("10", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.llms import Tongyi
    from langchain_core.prompts import PromptTemplate

    prompt_template = PromptTemplate.from_template(
        "我的邻居姓{lastname}，刚生了{gender}，你帮我起个名字，简单回答。"
    )

    model = Tongyi(model="qwen-max")
    chain = prompt_template | model

    res = chain.invoke({"lastname": "张", "gender": "女儿"})
    print(res)


## 11. 11FewShot提示词模板

- 对应源码：`11FewShot提示词模板.py`
- 本节说明：通过给示例的方式约束模型输出，观察 Few-shot Prompt 的组织结构。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("11", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.llms import Tongyi
    from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

    example_template = PromptTemplate.from_template("单词：{word}，反义词：{antonym}")

    examples_data = [
        {"word": "大", "antonym": "小"},
        {"word": "上", "antonym": "下"},
    ]

    few_shot_template = FewShotPromptTemplate(
        example_prompt=example_template,
        examples=examples_data,
        prefix="告知我单词的反义词，我提供如下的示例：",
        suffix="基于前面的示例告知我，{input_word} 的反义词是？",
        input_variables=["input_word"],
    )

    prompt_text = few_shot_template.format(input_word="左")
    print(prompt_text)

    model = Tongyi(model="qwen-max")
    print(model.invoke(prompt_text))


## 12. 12模板类的format和invoke方法

- 对应源码：`12模板类的format和invoke方法.py`
- 本节说明：区分模板的 `format()` 与 `invoke()`，理解字符串模板和消息模板的返回值差异。
- 运行提示：直接可运行，无需额外前置条件。


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotPromptTemplate, PromptTemplate

template = PromptTemplate.from_template("我的邻居是：{lastname}，最喜欢：{hobby}")

res = template.format(lastname="张大明", hobby="钓鱼")
print(res, type(res))

res2 = template.invoke({"lastname": "周杰轮", "hobby": "唱歌"})
print(res2, type(res2))


few_shot_template = FewShotPromptTemplate(
    example_prompt=PromptTemplate.from_template("问题：{question}，答案：{answer}"),
    examples=[
        {"question": "1+1等于几？", "answer": "2"},
        {"question": "中国的首都是哪？", "answer": "北京"},
    ],
    prefix="参考下面示例：",
    suffix="问题：{input_question}，答案：",
    input_variables=["input_question"],
)

res3 = few_shot_template.format(input_question="太阳从哪边升起？")
print(res3, type(res3))


chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个 helpful assistant。"),
        ("human", "{question}"),
    ]
)

res4 = chat_template.invoke({"question": "你是谁？"})
print(res4, type(res4))


## 13. 13ChatPromptTemplate的使用

- 对应源码：`13ChatPromptTemplate的使用.py`
- 本节说明：演示 `ChatPromptTemplate + MessagesPlaceholder` 的标准组合，这是构建会话类应用的高频写法。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("13", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

    chat_prompt_template = ChatPromptTemplate.from_messages(
        [
            ("system", "你是一个边塞诗人，可以作诗。"),
            MessagesPlaceholder(variable_name="history"),
            ("human", "请再来一首唐诗。"),
        ]
    )

    history_data = [
        ("human", "你来写一个唐诗。"),
        ("ai", "床前明月光，疑是地上霜，举头望明月，低头思故乡。"),
        ("human", "好诗再来一个。"),
        ("ai", "锄禾日当午，汗滴禾下土，谁知盘中餐，粒粒皆辛苦。"),
    ]

    prompt_value = chat_prompt_template.invoke({"history": history_data})
    print(prompt_value.to_string())

    model = ChatTongyi(model="qwen3-max")
    res = (chat_prompt_template | model).invoke({"history": history_data})

    print(res.content, type(res))


## 14. 14Chain的基础使用

- 对应源码：`14Chain的基础使用.py`
- 本节说明：把 Prompt 与 Model 通过 LCEL 串起来，体验最基本的链式调用。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("14", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

    chat_prompt_template = ChatPromptTemplate.from_messages(
        [
            ("system", "你是一个边塞诗人，可以作诗。"),
            MessagesPlaceholder(variable_name="history"),
            ("human", "请再来一首唐诗。"),
        ]
    )

    history_data = [
        ("human", "你来写一个唐诗。"),
        ("ai", "床前明月光，疑是地上霜，举头望明月，低头思故乡。"),
        ("human", "好诗再来一个。"),
        ("ai", "锄禾日当午，汗滴禾下土，谁知盘中餐，粒粒皆辛苦。"),
    ]

    model = ChatTongyi(model="qwen3-max")
    chain = chat_prompt_template | model

    for chunk in chain.stream({"history": history_data}):
        print(chunk.content, end="", flush=True)


## 15. 15[扩展]Python的或运算符的重写

- 对应源码：`15[扩展]Python的或运算符的重写.py`
- 本节说明：用纯 Python 理解 `|` 运算符重载，帮助从语言层面理解 LangChain 链式语法。
- 运行提示：直接可运行，无需额外前置条件。


In [ ]:
class Test:
    def __init__(self, name):
        self.name = name

    def __or__(self, other):
        return MySequence(self, other)

    def __str__(self):
        return self.name


class MySequence:
    def __init__(self, *args):
        self.sequence = list(args)

    def __or__(self, other):
        self.sequence.append(other)
        return self

    def run(self):
        for item in self.sequence:
            print(item)


if __name__ == "__main__":
    a = Test("a")
    b = Test("b")
    c = Test("c")
    e = Test("e")
    f = Test("f")
    g = Test("g")

    d = a | b | c | e | f | g
    d.run()
    print(type(d))


## 16. 16Runnable接口源码查看

- 对应源码：`16Runnable接口源码查看.py`
- 本节说明：通过最小示例观察 `Runnable` 组合后的对象类型。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("16", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.llms import Tongyi
    from langchain_core.prompts import PromptTemplate

    prompt = PromptTemplate.from_template("请用一句话介绍 {topic}")
    model = Tongyi(model="qwen-max")

    chain = prompt | model

    print(type(chain))
    print(chain.invoke({"topic": "LangChain"}))


## 17. 17StrOutputParser解析器

- 对应源码：`17StrOutputParser解析器.py`
- 本节说明：把聊天模型输出直接解析成字符串，是最常见的输出解析器之一。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("17", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate

    parser = StrOutputParser()
    model = ChatTongyi(model="qwen3-max")

    prompt = ChatPromptTemplate.from_messages(
        [
            ("human", "我邻居姓：{lastname}，刚生了{gender}，请起名，仅告知我名字无需其它内容。"),
        ]
    )

    chain = prompt | model | parser

    res = chain.invoke({"lastname": "张", "gender": "女儿"})
    print(res)
    print(type(res))


## 18. 18JsonOutputParser解析器

- 对应源码：`18JsonOutputParser解析器.py`
- 本节说明：演示如何让模型按 JSON 格式返回，并把结构化结果继续送入下游链路。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("18", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from pydantic import BaseModel, Field
    from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
    from langchain_core.prompts import PromptTemplate
    from langchain_core.runnables import RunnableLambda
    from langchain_community.chat_models.tongyi import ChatTongyi


    class NamePayload(BaseModel):
        name: str = Field(description="只返回 1 个中文名字")


    str_parser = StrOutputParser()
    json_parser = JsonOutputParser(pydantic_object=NamePayload)
    model = ChatTongyi(model="qwen3-max")

    first_prompt = PromptTemplate.from_template(
        "我邻居姓：{lastname}，刚生了{gender}，请帮忙起名字，"
        "并封装为 JSON 格式返回给我。"
        "{format_instructions}"
    ).partial(format_instructions=json_parser.get_format_instructions())

    second_prompt = PromptTemplate.from_template(
        "姓名：{name}，请帮我解析含义。"
    )

    def normalize_name(payload: dict) -> dict:
        if "name" in payload and payload["name"]:
            return {"name": payload["name"]}
        if "name_suggestions" in payload and payload["name_suggestions"]:
            return {"name": payload["name_suggestions"][0]}
        raise ValueError(f"未从模型输出中解析到 name 字段，实际返回：{payload}")

    chain = (
        first_prompt
        | model
        | json_parser
        | RunnableLambda(normalize_name)
        | second_prompt
        | model
        | str_parser
    )

    for chunk in chain.stream({"lastname": "张", "gender": "女儿"}):
        print(chunk, end="", flush=True)



## 19. 19RunnableLambda的基础使用

- 对应源码：`19RunnableLambda的基础使用.py`
- 本节说明：利用 `RunnableLambda` 在链路中做轻量数据转换。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("19", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnableLambda

    model = ChatTongyi(model="qwen3-max")
    str_parser = StrOutputParser()

    first_prompt = ChatPromptTemplate.from_messages(
        [
            ("human", "我邻居姓：{lastname}，刚生了{gender}，请帮忙起名字，仅生成一个名字，不要额外信息。"),
        ]
    )

    second_prompt = ChatPromptTemplate.from_messages(
        [
            ("human", "姓名：{name}，请帮我解析含义。"),
        ]
    )

    extract_name = RunnableLambda(lambda ai_msg: {"name": ai_msg.content.strip()})

    chain = first_prompt | model | extract_name | second_prompt | model | str_parser

    for chunk in chain.stream({"lastname": "曹", "gender": "女孩"}):
        print(chunk, end="", flush=True)


## 20. 20临时会话记忆

- 对应源码：`20临时会话记忆.py`
- 本节说明：使用内存型会话历史，让模型能记住同一个 session 内的上下文。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("20", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_core.chat_history import InMemoryChatMessageHistory
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_core.runnables.history import RunnableWithMessageHistory

    model = ChatTongyi(model="qwen3-max")

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "你需要结合历史对话回答用户的问题。"),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ]
    )

    str_parser = StrOutputParser()


    def print_prompt(full_prompt):
        print("=" * 20, full_prompt.to_string(), "=" * 20)
        return full_prompt


    base_chain = prompt | print_prompt | model | str_parser

    store = {}


    def get_history(session_id: str):
        if session_id not in store:
            store[session_id] = InMemoryChatMessageHistory()
        return store[session_id]


    conversation_chain = RunnableWithMessageHistory(
        base_chain,
        get_history,
        input_messages_key="input",
        history_messages_key="chat_history",
    )


    if __name__ == "__main__":
        session_config = {
            "configurable": {
                "session_id": "user_001",
            }
        }

        print(conversation_chain.invoke({"input": "小明有2只猫。"}, config=session_config))
        print(conversation_chain.invoke({"input": "小刚有1只狗。"}, config=session_config))
        print(conversation_chain.invoke({"input": "总共有几个宠物？"}, config=session_config))


## 21. 21长期会话记忆

- 对应源码：`21长期会话记忆.py`
- 本节说明：把历史消息落盘到文件，实现跨进程的会话持久化。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("21", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    import json
    import os
    from typing import Sequence

    from langchain_community.chat_models import ChatTongyi
    from langchain_core.chat_history import BaseChatMessageHistory
    from langchain_core.messages import BaseMessage, message_to_dict, messages_from_dict
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_core.runnables.history import RunnableWithMessageHistory


    class FileChatMessageHistory(BaseChatMessageHistory):
        def __init__(self, session_id: str, storage_path: str):
            self.session_id = session_id
            self.storage_path = storage_path
            self.file_path = os.path.join(self.storage_path, f"{self.session_id}.json")
            os.makedirs(self.storage_path, exist_ok=True)

        def add_messages(self, messages: Sequence[BaseMessage]) -> None:
            all_messages = list(self.messages)
            all_messages.extend(messages)

            payload = [message_to_dict(message) for message in all_messages]
            with open(self.file_path, "w", encoding="utf-8") as f:
                json.dump(payload, f, ensure_ascii=False, indent=2)

        @property
        def messages(self) -> list[BaseMessage]:
            try:
                with open(self.file_path, "r", encoding="utf-8") as f:
                    messages_data = json.load(f)
                return messages_from_dict(messages_data)
            except FileNotFoundError:
                return []

        def clear(self) -> None:
            with open(self.file_path, "w", encoding="utf-8") as f:
                json.dump([], f, ensure_ascii=False)


    model = ChatTongyi(model="qwen3-max")

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "你需要结合历史对话回答用户的问题。"),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ]
    )

    str_parser = StrOutputParser()


    def print_prompt(full_prompt):
        print("=" * 20, full_prompt.to_string(), "=" * 20)
        return full_prompt


    base_chain = prompt | print_prompt | model | str_parser


    def get_history(session_id: str):
        return FileChatMessageHistory(session_id, "./chat_history")


    conversation_chain = RunnableWithMessageHistory(
        base_chain,
        get_history,
        input_messages_key="input",
        history_messages_key="chat_history",
    )


    if __name__ == "__main__":
        session_config = {
            "configurable": {
                "session_id": "user_001",
            }
        }

        print(conversation_chain.invoke({"input": "小明有2只猫。"}, config=session_config))
        print(conversation_chain.invoke({"input": "小刚有1只狗。"}, config=session_config))
        print(conversation_chain.invoke({"input": "总共有几个宠物？"}, config=session_config))


## 22. 22CSVLoader的使用

- 对应源码：`22CSVLoader的使用.py`
- 本节说明：从 CSV 文件加载文档对象，熟悉 `Document` 的结构。
- 运行提示：直接可运行，无需额外前置条件。


In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path=str(DATA_DIR / "stu.csv"),
    csv_args={
        "delimiter": ",",
        "quotechar": '"',
    },
    encoding="utf-8",
)

for document in loader.lazy_load():
    print(document)


## 23. 23JSONLoader的使用

- 对应源码：`23JSONLoader的使用.py`
- 本节说明：从 JSON Lines 文件加载文档，需要 `jq` 包支持。
- 运行提示：运行前置：当前环境没有安装 jq。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("23", HAS_JQ, "当前环境没有安装 jq"):
    from langchain_community.document_loaders import JSONLoader

    loader = JSONLoader(
        file_path=str(DATA_DIR / "stu_json_lines.json"),
        jq_schema=".",
        text_content=False,
        json_lines=True,
    )

    documents = loader.load()
    for document in documents:
        print(document)


## 24. 24PyPDFLoader的使用

- 对应源码：`24PyPDFLoader的使用.py`
- 本节说明：从 PDF 中提取文本，体验受密码保护 PDF 的读取方式。
- 运行提示：运行前置：当前环境没有安装 pypdf。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("24", HAS_PYPDF, "当前环境没有安装 pypdf"):
    from langchain_community.document_loaders import PyPDFLoader

    loader = PyPDFLoader(
        file_path=str(DATA_DIR / "pdf2.pdf"),
        mode="single",
        password="itheima",
    )

    documents = loader.load()
    for index, doc in enumerate(documents, start=1):
        print(doc)
        print("=" * 20, index)


## 25. 25TextLoader和文档分割器

- 对应源码：`25TextLoader和文档分割器.py`
- 本节说明：先加载长文本，再切分成适合向量化的片段。
- 运行提示：直接可运行，无需额外前置条件。


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader(str(DATA_DIR / "Python基础语法.txt"), encoding="utf-8")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "！", "？", ".", "!", "?", " ", ""],
    length_function=len,
)

split_docs = splitter.split_documents(docs)
print(len(split_docs))

for doc in split_docs:
    print("=" * 20)
    print(doc)
    print("=" * 20)


## 26. 26内存向量存储

- 对应源码：`26内存向量存储.py`
- 本节说明：在内存里构建向量库，适合理解最简版的相似度检索流程。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("26", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.document_loaders import CSVLoader
    from langchain_community.embeddings import DashScopeEmbeddings
    from langchain_core.vectorstores import InMemoryVectorStore

    vector_store = InMemoryVectorStore(
        embedding=DashScopeEmbeddings(model="text-embedding-v4")
    )

    loader = CSVLoader(
        file_path="./data/info.csv",
        encoding="utf-8",
        source_column="source",
    )

    documents = loader.load()
    ids = [f"id{i}" for i in range(1, len(documents) + 1)]

    vector_store.add_documents(documents=documents, ids=ids)
    vector_store.delete(ids=["id1", "id2"])

    result = vector_store.similarity_search("瑞达法", k=3)
    print(result)


## 27. 27外部向量持久化存储

- 对应源码：`27外部向量持久化存储.py`
- 本节说明：体验 Chroma 持久化向量库存储与检索。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("27", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_chroma import Chroma
    from langchain_community.embeddings import DashScopeEmbeddings

    vector_store = Chroma(
        collection_name="test",
        embedding_function=DashScopeEmbeddings(model="text-embedding-v1"),
        persist_directory="./chroma_db",
    )

    result = vector_store.similarity_search(
        "Python是不是简单易学呀",
        k=3,
        filter={"source": "黑马程序员"},
    )

    print(result)


## 28. 28向量检索构建提示词

- 对应源码：`28向量检索构建提示词.py`
- 本节说明：把检索结果拼接进提示词，形成最朴素的 RAG 问答链。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("28", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_community.embeddings import DashScopeEmbeddings
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.vectorstores import InMemoryVectorStore


    def print_prompt(prompt):
        print(prompt.to_string())
        print("=" * 20)
        return prompt


    model = ChatTongyi(model="qwen3-max")
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "请优先依据我提供的参考资料，简洁且专业地回答用户问题。\n参考资料：\n{context}"),
            ("human", "用户提问：{input}"),
        ]
    )

    vector_store = InMemoryVectorStore(
        embedding=DashScopeEmbeddings(model="text-embedding-v1")
    )

    vector_store.add_texts(
        [
            "减肥就是要少吃多练。",
            "在减脂期间吃东西很重要，清淡少油、控制卡路里摄入并运动起来。",
            "跑步是很好的运动哦。",
        ]
    )

    input_text = "怎么减肥？"
    result = vector_store.similarity_search(input_text, k=2)
    reference_text = "\n".join(f"- {doc.page_content}" for doc in result)

    chain = prompt | print_prompt | model | StrOutputParser()

    res = chain.invoke({"input": input_text, "context": reference_text})
    print(res)


## 29. 29RunnablePassthrough的使用

- 对应源码：`29RunnablePassthrough的使用.py`
- 本节说明：用 `RunnablePassthrough` 把用户输入与检索结果并行组织起来，形成更标准的 RAG 写法。
- 运行提示：运行前置：未设置 DASHSCOPE_API_KEY。若条件不满足，本节会自动跳过。


In [ ]:
if section_ready("29", HAS_DASHSCOPE, "未设置 DASHSCOPE_API_KEY"):
    from langchain_community.chat_models import ChatTongyi
    from langchain_community.embeddings import DashScopeEmbeddings
    from langchain_core.documents import Document
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnablePassthrough
    from langchain_core.vectorstores import InMemoryVectorStore


    def print_prompt(prompt):
        print(prompt.to_string())
        print("=" * 20)
        return prompt


    model = ChatTongyi(model="qwen3-max")
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "请优先依据我提供的参考资料，简洁且专业地回答用户问题。\n参考资料：\n{context}"),
            ("human", "用户提问：{input}"),
        ]
    )

    vector_store = InMemoryVectorStore(
        embedding=DashScopeEmbeddings(model="text-embedding-v1")
    )

    vector_store.add_texts(
        [
            "减肥就是要少吃多练。",
            "在减脂期间吃东西很重要，清淡少油、控制卡路里摄入并运动起来。",
            "跑步是很好的运动哦。",
        ]
    )

    retriever = vector_store.as_retriever(search_kwargs={"k": 2})


    def format_docs(docs: list[Document]) -> str:
        if not docs:
            return "无相关参考资料"

        return "\n".join(f"- {doc.page_content}" for doc in docs)


    chain = (
        {
            "input": RunnablePassthrough(),
            "context": retriever | format_docs,
        }
        | prompt
        | print_prompt
        | model
        | StrOutputParser()
    )

    res = chain.invoke("怎么减肥？")
    print(res)
